# Session 6: Deploying MCP Servers

**Learning goals:**

- Configure local and remote MCP servers in JSON, discover their tools and route calls to the owning client.
- Deploy a FastMCP server and verify its Streamable HTTP endpoint with an MCP client.
- Deploy a FastAPI web chat that calls the remote MCP server and keeps separate browser histories.

**Prerequisites:**

- [Session 5: Building MCP Server & Client](<../05-Building-MCP-Server-and-Client/Session 5 Building MCP Server & Client.ipynb>) and familiarity with Python, a terminal and virtual environments.
- A local copy of the practice files and a Python environment with the practice dependencies.
- For the interactive chatbot and web chat, a Gemini or OpenRouter API key and a model that supports tool calling. For deployment, a WeatherAPI key, a Render account and a Git repository that Render can access.

Build a chatbot that drafts notices for a fictional TAFE new-student orientation. Use internet access to try the public Microsoft Learn MCP server. Section 1 uses the same simulated weather records as [Adding Prompt and Resource Features in Session 5](<../05-Building-MCP-Server-and-Client/Session 5 Building MCP Server & Client.ipynb#adding-prompt-and-resource-features>). Section 2 deploys the Weather MCP server with live WeatherAPI observations. The Microsoft Learn MCP server is public and requires no Microsoft sign-in or API key.


## Outline

1. [Connect Multiple MCP Servers](#connect-multiple-mcp-servers)
2. [Deploy Remote MCP Server](#deploy-remote-mcp-server)
3. [Deploy Web Chat](#deploy-web-chat)


## 1. Connect Multiple MCP Servers

### 1.1 How Multiple MCP Servers Connect

The chatbot acts as the MCP host. It creates one client per server, then uses each client to discover and call that server's tools. A local client can exchange messages with a subprocess over stdio (standard input and output). A remote client can exchange messages with an MCP endpoint over Streamable HTTP. Both transports can be active in one chatbot turn. [MCP architecture overview](https://modelcontextprotocol.io/docs/learn/architecture)

![An AI host contains two MCP clients. One connects to a weather server over stdio, and the other connects to a document server over HTTP.](images/mcp-architecture.png)

The host also communicates with the LLM provider over a separate HTTPS connection. It sends the question and allowed tool definitions to the model. The model can request a tool by name and supply JSON arguments. The host routes the request through the appropriate MCP client and returns the result to the model.

For reference implementations, see the [MCP project’s server repository](https://github.com/modelcontextprotocol/servers). To search by category, use the [Awesome MCP Servers directory](https://mcpservers.org/).

### 1.2 Configure MCP Client Connections

This exercise connects to three MCP servers.

- **Weather (`weather`)** runs locally over stdio and returns simulated weather for Perth or Sydney.
- **Filesystem (`files`)** runs locally over stdio and reads the bundled [orientation plan](sample_data/event-plan.md). See the [course Filesystem server implementation](practice/filesystem_mcp_server.py).
- **Microsoft Learn (`learn`)** connects over Streamable HTTP and searches Microsoft's documentation for Teams guidance. See the [Microsoft Learn MCP server overview](https://learn.microsoft.com/en-us/training/support/mcp).

The host reads [mcp_servers.json](practice/mcp_servers.json) and creates one client per entry. `command` and `args` start local subprocesses in the `practice` directory. `url` connects to a remote MCP endpoint over Streamable HTTP.

```json
{
  "mcpServers": {
    "weather": {
      "command": "python",
      "args": ["sample_weather_mcp_server.py", "--transport", "stdio"]
    },
    "files": {
      "command": "python",
      "args": ["filesystem_mcp_server.py"]
    },
    "learn": {
      "url": "https://learn.microsoft.com/api/mcp"
    }
  }
}
```

In [multi_server_chatbot.py](practice/multi_server_chatbot.py), `create_client()` chooses the transport for each entry. The host stores the clients by server name.

```python
def create_client(settings):
    """Use Streamable HTTP for a URL or stdio for a local command."""
    if "url" in settings:
        return Client(settings["url"])

    command = settings.get("command", "python")
    # Keep course scripts in the same Python environment as the host.
    executable = sys.executable if command == "python" else command
    return Client(StdioTransport(
        command=executable,
        args=settings.get("args", []),
        cwd=str(PRACTICE_DIR),
        keep_alive=False,
    ))

# Inside MultiServerChatbot.__init__, build one client for each configured server.
self.clients = {label: create_client(settings) for label, settings in self.servers.items()}
```

The following method uses the single-server connection in Session 6's [local chatbot module](practice/mcp_chatbot.py) to open each client and discover its tools. It gives each selected tool a server prefix such as `weather_get_weather` and stores the client and original tool name for routing. `AsyncExitStack` keeps the connections open during the chat and closes them if a later connection fails.

```python
from contextlib import AsyncExitStack, asynccontextmanager

# Method of MultiServerChatbot. EXERCISE_TOOLS selects the case-study tools.
@asynccontextmanager
async def connect_to_servers(self):
    """Reuse the single-server connection for each configured client."""
    async with AsyncExitStack() as stack:
        tools_for_llm = []
        routes = {}
        for label, client in self.clients.items():
            tools = await stack.enter_async_context(self.connect_to_server(client, server_label=label))
            exercise_tool = EXERCISE_TOOLS.get(label)
            if exercise_tool and exercise_tool not in {tool["function"]["name"] for tool in tools}:
                raise ValueError(f"The {label} server is missing {exercise_tool}.")
            for tool in tools:
                function = tool["function"]
                original_name = function["name"]
                if exercise_tool and original_name != exercise_tool:
                    continue
                # Prefix names to keep tools from different servers distinct.
                name = f"{label}_{original_name}"
                if name in routes:
                    raise ValueError(f"Duplicate chatbot tool name: {name}")
                routes[name] = (client, original_name)
                tools_for_llm.append({**tool, "function": {**function, "name": name}})
        self.available_tools = tools_for_llm
        self.routes = routes
        yield
```


### 1.3 Wet-Weather Orientation Case Study

A TAFE organiser needs a draft notice for Sydney's new-student orientation. The [event plan](sample_data/event-plan.md) gives the rain rule and backup venue. Simulated light rain triggers the indoor plan, and [Microsoft Learn](https://learn.microsoft.com/en-us/training/support/mcp) supplies Teams guidance for attendees without a TAFE account.

![Sequence diagram. The MCP host sends the prompt and tool definitions to the LLM API over HTTPS. It routes event-plan and weather calls over stdio and the Teams documentation query over Streamable HTTP. All results return through the host to the LLM before it drafts the notice.](images/orientation-mcp-sequence.png)

The host returns each tool result to the LLM API with the original `tool_call_id`. For Sydney, the expected draft gives **Student Centre, Room S3 at 10 am local time** and cites Microsoft Learn. Perth's simulated partly cloudy conditions keep the **campus courtyard at 10 am local time**.


### 1.4 Run in Terminal

[Download Session 6 practice and deployment files](practice/session-6-multi-mcp.zip){download="session-6-multi-mcp.zip"}. Use this ZIP for sections 1.4, 2 and 3. Extract it and open a terminal in `06-Deploying-MCP-Servers`.

Install the practice dependencies.

```bash
python -m pip install -r practice/requirements.txt
```

Open the included `practice/.env` in a text editor. The ZIP leaves every value blank. For OpenRouter, enter `openrouter` for `LLM_PROVIDER`, `openrouter/free` or another model that supports tool calling for `CHAT_MODEL`, and your `OPENROUTER_API_KEY`. This terminal exercise uses bundled weather samples. Fill in `WEATHER_API_KEY` when you reach section 2.2.

```dotenv
LLM_PROVIDER=
CHAT_MODEL=
OPENROUTER_API_KEY=
GEMINI_API_KEY=
WEATHER_API_KEY=
```

Keep `.env` local and out of Git. An exported terminal variable takes precedence over the same entry in `.env`. To use Gemini, change `LLM_PROVIDER` to `gemini`, set `CHAT_MODEL` to a Gemini model that supports tool calling and fill in `GEMINI_API_KEY`. [OpenRouter Free Models Router](https://openrouter.ai/openrouter/free/providers)

Start the chatbot.

```bash
python practice/multi_server_chatbot.py
```

Ask **Draft the Sydney orientation notice using the event plan, simulated weather and Microsoft Learn guidance on joining Teams without a TAFE account. Include the source URL.** Look for tool calls to Filesystem, Weather and Microsoft Learn in the terminal.

After the notice appears, enter `/save` followed by a path outside the course repository. For example, type this at the chatbot's `You:` prompt.

```text
/save ~/sydney-orientation-notice.md
```

This writes the latest reply to a new Markdown file. Save the Sydney draft before asking for the Perth draft. Type `quit` to exit.


## 2. Deploy Remote MCP Server

Publish the Weather MCP server's source on GitHub, then connect the repository to Render. Render runs the server and exposes its Streamable HTTP endpoint.

### 2.1 Prepare GitHub Repository

Git records changes to your project as commits in a local repository. GitHub hosts a copy of that repository online. In this exercise, you use Git to commit the deployment files and push them to GitHub. Render then builds the service from the `main` branch of your GitHub repository.

If you are new to Git, start with the [official Git tutorial](https://git-scm.com/docs/gittutorial#importing-a-new-project) on initialising a repository, staging files and making your first commit.

1. [Download the Session 6 deployment ZIP](practice/session-6-multi-mcp.zip){download="session-6-multi-mcp.zip"}. Extract it into a new folder and open a terminal at the level containing `.gitignore`, `LICENSE` and `06-Deploying-MCP-Servers`. The package includes the Weather MCP server and web chat files for both Render services.
2. Sign in to [GitHub](https://github.com/) and select **+ → New repository**.

   - Choose your own account as **Owner**, enter a **Repository name** and select the visibility.
   - Leave the README, licence and `.gitignore` initialisation options off, then select **Create repository**.
   - On the **Quick setup** page, select **HTTPS** and copy the repository URL. [Create a repository on GitHub](https://docs.github.com/en/repositories/creating-and-managing-repositories/creating-a-new-repository#creating-a-new-repository-from-the-web-ui)

3. Set up HTTPS authentication. In GitHub, go to **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**.

   - Name the token, set an expiry and select your own account as **Resource owner**.
   - Under **Repository access**, choose **Only select repositories** and select the repository you just created. Under **Permissions**, give **Contents** **Read and write** access.
   - Select **Generate token** and save the token securely. Keep it out of the repository URL, project files and commits. [GitHub guide to personal access tokens](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens#creating-a-fine-grained-personal-access-token)

4. Run the commands below. Replace `YOUR_GITHUB_REPOSITORY_URL` with the HTTPS URL of your new repository. Keep the chatbot API keys in `practice/.env`, which the included `.gitignore` excludes from Git.

```bash
git init -b main
git add .
git commit -m "feat: add Session 6 MCP deployment files"
git remote add origin "YOUR_GITHUB_REPOSITORY_URL"
git push -u origin main
```

If Git prompts for credentials during `git push`, enter your GitHub username at **Username** and paste your personal access token at **Password**. [Use a personal access token on the command line](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens#using-a-personal-access-token-on-the-command-line)

On GitHub, confirm that `main` contains the Session 6 `practice` and `sample_data` folders. [GitHub guide to pushing local code](https://docs.github.com/en/migrations/importing-source-code/using-the-command-line-to-import-source-code/adding-locally-hosted-code-to-github)


### 2.2 Deploy Weather Server on Render

With `--live`, the Weather MCP server calls WeatherAPI's current conditions endpoint. The returned temperature, conditions and observation time can change between requests. [WeatherAPI documentation](https://www.weatherapi.com/docs/)

#### Test Streamable HTTP locally

Set `WEATHER_API_KEY` in `practice/.env` or export it in your terminal. From `06-Deploying-MCP-Servers`, install the practice dependencies if needed, then start the [Weather MCP server](practice/sample_weather_mcp_server.py).

```bash
python -m pip install -r practice/requirements.txt
python practice/sample_weather_mcp_server.py --live --host 127.0.0.1 --port 8000
```

Open [the local health endpoint](http://127.0.0.1:8000/health) and check for `OK`. Keep the server running. In a second terminal with Node.js 22.19.0 or newer, launch [MCP Inspector](https://modelcontextprotocol.io/docs/tools/inspector).

```bash
npx @modelcontextprotocol/inspector
```

1. In Inspector, select **Streamable HTTP**, enter `http://127.0.0.1:8000/mcp` and select **Connect**.
2. In **Tools**, list the tools and select `get_weather`. Set `city` to `Perth` and run the tool.
3. Check the temperature, conditions and observation time. The source should be `WeatherAPI (current conditions)`. The Weather MCP server reads the WeatherAPI key from its own environment.

Keep the local server running for the web chat test in section 3.

#### Create Render Web Service

1. In the Render Dashboard, choose **New → Web Service** and connect GitHub if prompted.
2. Select the GitHub repository from section 2.1 and choose the `main` branch.
3. Apply the settings below. Under **Environment Variables**, add `WEATHER_API_KEY` with your WeatherAPI key, then deploy. Set the health check path in the service settings.

| Setting | Value |
|---|---|
| Name | A unique name for your weather service |
| Branch | `main` |
| Language | Python 3 |
| Root Directory | `06-Deploying-MCP-Servers` |
| Build Command | `python -m pip install -r practice/requirements.txt` |
| Start Command | `python practice/sample_weather_mcp_server.py --live --host 0.0.0.0 --port "$PORT"` |
| Health Check Path | `/health` |

Render runs both commands from the selected root directory. The server binds to `0.0.0.0` and listens on the `PORT` that Render supplies.

For an existing service, update its **Start Command** and add the key under **Environment → Add Environment Variable**. Select **Save and deploy** to apply the new value. [Render environment variables](https://render.com/docs/configure-environment-variables)

[Render Web Services](https://render.com/docs/web-services) · [Render root directory settings](https://render.com/docs/monorepo-support) · [Render health checks](https://render.com/docs/health-checks)

#### Verify deployed MCP endpoint

When the deployment is live, copy the Weather MCP service's HTTPS URL from Render. Open it with `/health` appended and check for `OK`. To open MCP Inspector for the deployed server, replace `YOUR-WEATHER-SERVICE.onrender.com` with the hostname Render gives you and run the following command.

```bash
npx @modelcontextprotocol/inspector --server-url https://YOUR-WEATHER-SERVICE.onrender.com/mcp --transport http
```

In Inspector, connect to the server and run `get_weather` with `city` set to `Perth`. Check that it returns current WeatherAPI observations. Keep the full HTTPS `/mcp` URL for section 3. [MCP Inspector documentation](https://modelcontextprotocol.io/docs/tools/inspector)


## 3. Deploy Web Chat

Continue with the GitHub repository you created in [section 2.1](#prepare-github-repository). Both Render services use the `main` branch of that repository. For the local test, use the files you already extracted from the Session 6 ZIP.

[The FastAPI web chat](practice/web_chat.py) connects to the weather MCP server over Streamable HTTP and runs the Filesystem server as a subprocess. Test the browser, FastAPI and both MCP servers locally before deploying the web chat as a second Render service.

| Connection | Interface |
|---|---|
| Browser → web chat service | Same-origin `POST /chat` |
| Web chat service → LLM provider | HTTPS chat completions |
| Web chat service → weather service | Streamable HTTP at the weather server's `/mcp` URL |
| Web chat service → Filesystem server | stdio to the bundled subprocess |

### Test the web chat locally

The [web interface](practice/index.html) follows Session 5's design. Use the example prompts to fill the message box, **New chat** to start a separate conversation and **History** to reopen an earlier one. The web service serves the page at `GET /`, lists or opens conversations at `GET /chats` and `GET /chats/{chat_id}`, and handles turns at `POST /chat`.

Keep the local weather server from section 2 running. In a new terminal, export the provider variables from section 1. From `06-Deploying-MCP-Servers` in Bash or Zsh, `set -a; source practice/.env; set +a` loads them into the terminal. Set `MCP_SERVER_URL` to `http://127.0.0.1:8000/mcp`. Leave `ENABLE_MICROSOFT_LEARN` unset for this two-server deployment exercise. From the session directory, start FastAPI.

```bash
python -m uvicorn web_chat:app --app-dir practice --host 127.0.0.1 --port 8080
```

Open [the local web chat](http://127.0.0.1:8080/) and ask **Draft the Perth orientation notice using the event plan and current weather.** Check for the 10 am start time, current Perth conditions and a venue that matches the event plan. Ask **How long does it run?** to check that the browser can continue the conversation. Use **New chat** to start a second conversation, then reopen the Perth chat from **History**. To make Teams guidance available in the browser, set `ENABLE_MICROSOFT_LEARN` to `1` and restart the local web service.

### Create the Render web chat service

In the Render Dashboard, choose **New → Web Service** again. Select the GitHub repository you created in section 2.1 and choose `main`. Set **Root Directory** to `06-Deploying-MCP-Servers`. This creates a second Render service for the Web Chat alongside the Weather MCP service.

| Setting | Value |
|---|---|
| Name | A unique name for your web chat service |
| Branch | `main` |
| Language | Python 3 |
| Root Directory | `06-Deploying-MCP-Servers` |
| Build Command | `python -m pip install -r practice/requirements.txt` |
| Start Command | `python -m uvicorn web_chat:app --app-dir practice --host 0.0.0.0 --port "$PORT" --workers 1` |
| Health Check Path | `/health` |

[Deploy a FastAPI app on Render](https://render.com/docs/deploy-fastapi) · [Render root directory settings](https://render.com/docs/monorepo-support)


### Connect the two Render services

Before deploying the web chat, add the following variables in its **Environment** settings. If the service already exists, save the variables and deploy again.

| Variable | Value |
|---|---|
| `MCP_SERVER_URL` | The weather service's full HTTPS URL ending in `/mcp` |
| `LLM_PROVIDER` | `openrouter` or `gemini` |
| `CHAT_MODEL` | A model ID from the chosen provider that supports tool calling |
| `OPENROUTER_API_KEY` or `GEMINI_API_KEY` | The key for the chosen provider |
| `CHAT_ACCESS_PASSWORD` | A strong password you choose for browser access to the Web Chat, using ASCII characters |

The web chat uses its own HTTPS URL. Store the model provider key in the Web Chat service and `WEATHER_API_KEY` in the Weather MCP service.

Set `CHAT_ACCESS_PASSWORD` in the **Web Chat** service's Render environment. Render deployment requires this setting. When the browser prompts for credentials, enter `learner` as the username and the value you set as the password.

[Render environment variables](https://render.com/docs/configure-environment-variables)

### Verify the deployed web chat

1. Open the web chat service's HTTPS URL and sign in as `learner` using the classroom password.
2. Ask **Draft the Perth orientation notice using the event plan and current weather.** Check for the 10 am start time, WeatherAPI as the source and a venue that matches the reported conditions.
3. Ask **How long does it run?** in the same browser. Open a private window and sign in to check that it starts with a separate conversation.
4. If the chat fails, inspect the web service's Render logs and verify its model settings and the weather service's `/mcp` URL.

The single worker keeps browser histories in one process for the classroom demo. Histories reset when the web chat service restarts. Stop the two Render services when you finish the exercise.
